# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH03/ch03_TreeQuest.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/>
</a>

# 关于本 Notebook

本 Notebook 演示如何使用 [**TreeQuest**](https://github.com/SakanaAI/treequest) 配合 **AB-MCTS 风格搜索（AB-MCTS style search）**，迭代改进由 LLM 生成的答案。流程会先提出一个初始解，再对其进行改写优化，通过结构化评审器（structured judge）为每个候选答案打分，最后让蒙特卡洛树搜索（Monte Carlo Tree Search, MCTS）在有限搜索预算内保留目前找到的最佳结果。

## 你将看到什么

* **树引导优化（tree-guided refinement）**：使用一个只有两种动作的简单生成器——先生成初稿，再持续优化
* **结构化评审（structured judging）**：让 LLM 以 JSON 输出 `[0, 1]` 区间内的数值质量分
* **有状态搜索（stateful search）**：使用 `treequest.ABMCTSA`，定期跟踪当前最佳候选，并在最后执行 top-k 选择
* **角色清晰分离**：生成器（generator）、优化器（refiner）、评审器（judge）和搜索控制器（search controller）彼此独立

## 你将运行什么

1. 安装 `treequest[abmcts-m]`，并配置一个兼容 OpenAI 接口的 `OpenAI` 客户端。
2. 为 MCTS 节点定义最小化的 `State`：包含当前 `llm_answer` 及其 `score`。
3. 实现：

   * `initial_generation()`：生成第一版答案并打分
   * `refine_answer()`：改进已有答案并重新打分
   * `evaluate_answer()`：通过结构化响应评估答案质量
   * `generate()`：作为 TreeQuest 用来扩展节点的动作
4. 创建 `algo = tq.ABMCTSA()` 并运行一个短搜索循环，在过程中打印当前最佳候选。
5. 使用 `tq.top_k` 选择并打印最终最佳答案。

## 它如何工作

* **状态（State）**
  每个节点保存 `llm_answer` 和 `score`。MCTS 使用这个分数进行选择（selection）与反向传播（backpropagation）。

* **初始生成与优化步骤（Initial and refine steps）**
  `initial_generation()` 要求模型先为任务生成一个初始解。
  `refine_answer()` 要求模型在保留任务目标的同时，提高答案的清晰度与准确性。

* **评分（Scoring）**
  `evaluate_answer()` 要求模型返回类似 `{"score": 0.92}` 的 JSON 对象，并用 Pydantic schema 解析。把评审与生成分离，通常能提升评估稳定性。

* **TreeQuest**
  `ABMCTSA` 负责节点选择、通过 `generate` 扩展节点、执行 rollout（若有定义）以及价值反向传播。`generate` 会返回新的 `State`，以及父边对应的数值。这里直接把父节点分数作为边值；对于只做迭代优化的简单流程而言，这是一种朴素但可用的做法。

* **Top-k**
  搜索过程中定期调用 `tq.top_k` 查看当前最佳候选，结束时再调用一次以得到最终胜出答案。

## 如何扩展

* 加入**自一致性评审器（self-consistency judge）**，结合测试用例或风格规范进行评估。
* 使用**成对评审器（pairwise judge）**比较两个答案并返回偏好，再把偏好转换成供 MCTS 使用的分数。
* 加入 **rollout 步骤**，在正式评分前连续执行多次小幅优化。
* 在评审器里加入**领域测试（domain tests）**，例如实际运行代码并检查输出。
* 通过简单存储层持久化搜索树与执行轨迹，便于后续分析。

## 环境要求与说明

* 在 `OpenAI(api_key="...")` 中配置有效 API Key。
* 评审模型必须支持结构化响应。本示例使用 `client.chat.completions.parse` 配合 Pydantic schema。
* 生成阶段使用中等 temperature 以增加多样性；评审阶段使用较低 temperature 以提高稳定性。
* 本示例的搜索深度和步数都很小。实际使用时应谨慎增加，因为 API 成本会随调用次数上升。

# 安装依赖

In [ ]:
!pip install treequest==0.2.0

# API 配置

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
from openai import OpenAI
client = OpenAI()

# 导入依赖

In [ ]:
import json
import treequest as tq
from openai import OpenAI
from dataclasses import dataclass
from pydantic import BaseModel, Field

# 类与数据

搜索节点的载荷（payload）。
- `llm_answer`：当前候选答案
- `score`：由评审器（judge）给出的 `[0, 1]` 区间数值分数

In [ ]:
@dataclass
class State:
    llm_answer: str
    score: float


# 初始生成（Initial generation）

创建第一份候选答案并为其评分。
模型先从零开始生成一个初始解，再调用评审器得到分数。函数返回一个 `State`，MCTS 可以把它视作根节点的子节点。

In [ ]:
def initial_generation() -> State:
    prompt = "Q: Write code in Python for the Fibonacci sequence. \nA:"
    response = client.chat.completions.create(
        model="gpt-4o",  # 必须使用支持 JSON 模式的模型
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    answer = response.choices[0].message.content.strip()
    score = evaluate_answer(answer)
    return State(llm_answer=answer, score=score)

# 优化（Refinement）

改进已有答案并重新评分。优化提示词要求模型提升清晰度、准确性和完整性。函数返回一个新的 `State`，其中包含优化后的文本和最新分数。

In [ ]:
def refine_answer(llm_answer: str, score: float) -> State:
    prompt = f"""The current answer is:\n\n{llm_answer}\n\nPlease improve this answer to be more informative, accurate, and clear."""
    response = client.chat.completions.create(
        model="gpt-4o",  # 必须使用支持 JSON 模式的模型
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    refined = response.choices[0].message.content.strip()
    score = evaluate_answer(refined)
    return State(llm_answer=refined, score=score)

# 评审 Schema（Judge schema）

结构化评审响应。
- `score`：范围为 `[0, 1]` 的浮点数，越高越好。通过 Pydantic 强制约束，可尽早捕获解析失败。

In [ ]:
class ScoreResponse(BaseModel):
    score: float = Field(..., ge=0.0, le=1.0)

# 评审函数（Judging function）

要求 LLM 评审器返回只包含一个 `score` 键的 JSON 对象。这里使用 OpenAI 客户端把结构化结果解析为 `ScoreResponse`；发生异常时回退为 `0.5`，让搜索流程能够继续。

In [ ]:
def evaluate_answer(answer: str) -> float:
    prompt = (
        f"Evaluate the quality of this answer on a scale from 0 to 1.\n"
        f"Return a JSON object like {{\"score\": 0.92}}.\n\n"
        f"Answer:\n{answer}"
    )

    try:
        completion = client.chat.completions.parse(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            response_format=ScoreResponse,
        )
        return completion.choices[0].message.parsed.score
    except Exception as e:
        print(f"[Evaluation error] {e}")
        return 0.5

# TreeQuest 生成器（Generator for TreeQuest）

这是 TreeQuest 的节点扩展函数。若不存在 `parent_state`，就创建初始候选；否则继续优化父节点的答案。

In [ ]:
def generate(parent_state: State | None) -> tuple[State, float]:
    if parent_state is None:
        return initial_generation(), initial_generation().score
    return refine_answer(parent_state.llm_answer, parent_state.score), parent_state.score

# 搜索循环（Search loop）

In [ ]:
# TreeQuest 搜索循环
algo = tq.ABMCTSA()
search_tree = algo.init_tree()

# 先运行较少步数；真实场景中应逐步增加搜索预算
for i in range(5):
    search_tree = algo.step(search_tree, {'LLM-Refine': generate})
    # 定期查看当前最佳候选
    if (i + 1) % 5 == 0:
        best, _ = tq.top_k(search_tree, algo, k=1)[0]
        print(f"[Step {i+1}] Best so far: {best.llm_answer} (score={best.score:.2f})")

# 最终选择
best_state, _ = tq.top_k(search_tree, algo, k=1)[0]
print(f"\n Final Best Answer: {best_state.llm_answer} (score={best_state.score:.2f})")


[Step 5] Best so far: Certainly! Let's refine the explanation and the code to make it more informative, accurate, and clear.

### Improved Explanation

The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, typically starting with 0 and 1. The sequence starts as 0, 1, 1, 2, 3, 5, 8, and so on. In this code, we aim to generate the first `n` terms of the Fibonacci sequence.

The function `fibonacci(n)` is designed to return a list containing the first `n` terms of the Fibonacci sequence. Here's a step-by-step explanation:

1. **Function Definition**: The function `fibonacci(n)` takes one argument `n`, which represents the number of terms you want to generate.

2. **Initialization**: 
    - An empty list called `sequence` is initialized to store the Fibonacci numbers.
    - Two variables, `a` and `b`, are initialized to 0 and 1, representing the first two numbers in the Fibonacci sequence.

3. **While Loop**: 
    - The loop continues until t